In [17]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [18]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import KFold
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings("ignore")

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.10.0+cu128 | CUDA: True


In [19]:
class CFG:
    in_channels = 1
    num_classes = 1
    batch_size = 8
    num_epochs = 20          # Reduced for multiple experiments
    lr = 3e-4
    weight_decay = 1e-5
    n_folds = 5              # You can increase to 10 later
    seed = 42
    save_dir = "/kaggle/working"
    
    # Ablation Experiments
    experiments = [
        {"exp":1, "img_size":128, "depth":2, "name":"128x128_2-2"},
        {"exp":2, "img_size":64,  "depth":3, "name":"64x64_3-3"},
        {"exp":3, "img_size":32,  "depth":4, "name":"32x32_4-4"},
        {"exp":4, "img_size":16,  "depth":5, "name":"16x16_5-5"},
        {"exp":5, "img_size":8,   "depth":6, "name":"8x8_6-6"},
    ]

torch.manual_seed(CFG.seed)
np.random.seed(CFG.seed)

In [20]:
# BASE_DIR = Path("/kaggle/input")
# DATASET_PATH = BASE_DIR / "datasets" / "iamtapendu" / "chest-x-ray-lungs-segmentation" / "Chest-X-Ray" / "Chest-X-Ray"

# IMG_DIR = DATASET_PATH / "image"
# MASK_DIR = DATASET_PATH / "mask"

# def collect_pairs(img_dir, mask_dir):
#     pairs = []
#     for img_path in sorted(list(Path(img_dir).glob("*.png")) + list(Path(img_dir).glob("*.jpg"))):
#         stem = img_path.stem
#         mask_path = Path(mask_dir) / f"{stem}.png"
#         if not mask_path.exists():
#             mask_path = Path(mask_dir) / f"{stem}.jpg"
#         if mask_path.exists():
#             pairs.append((str(img_path), str(mask_path)))
#     return pairs

# pairs = collect_pairs(IMG_DIR, MASK_DIR)
# print(f"Total image-mask pairs: {len(pairs)}")

In [21]:
# =============================================
# CELL 3: Dataset Paths & Pair Collection (CORRECTED)
# =============================================

from pathlib import Path

BASE_DIR = Path("/kaggle/input")
DATASET_PATH = BASE_DIR / "datasets" / "iamtapendu" / "chest-x-ray-lungs-segmentation" / "Chest-X-Ray" / "Chest-X-Ray"

IMG_DIR  = DATASET_PATH / "image"
MASK_DIR = DATASET_PATH / "mask"

print("Dataset Path Verification:")
print(f"Image folder : {IMG_DIR}")
print(f"Mask folder  : {MASK_DIR}")
print(f"Image folder exists: {IMG_DIR.exists()}")
print(f"Mask folder exists : {MASK_DIR.exists()}")

# ==================== IMPROVED collect_pairs ====================
def collect_pairs(img_dir, mask_dir):
    img_dir = Path(img_dir)
    mask_dir = Path(mask_dir)
    pairs = []
    
    # Get all images
    image_files = sorted(list(img_dir.glob("*.png")) + 
                        list(img_dir.glob("*.jpg")) + 
                        list(img_dir.glob("*.jpeg")))
    
    print(f"Found {len(image_files)} image files")
    
    for img_path in image_files:
        stem = img_path.stem
        
        # Try different mask extensions
        mask_path = mask_dir / f"{stem}.png"
        if not mask_path.exists():
            mask_path = mask_dir / f"{stem}.jpg"
        if not mask_path.exists():
            mask_path = mask_dir / f"{stem}.jpeg"
        
        if mask_path.exists():
            pairs.append((str(img_path), str(mask_path)))
        else:
            print(f"Warning: Mask not found for {stem}")
    
    return pairs

# Load pairs
pairs = collect_pairs(IMG_DIR, MASK_DIR)
print(f"\n✅ Total matched image-mask pairs: {len(pairs)}")

if len(pairs) == 0:
    raise FileNotFoundError("No matching image-mask pairs found! Check folder structure.")

# Create DataFrame for easy viewing
df = pd.DataFrame(pairs, columns=["image", "mask"])
print("\nFirst 5 pairs:")
print(df.head())

# Optional: Save pairs list
df.to_csv("/kaggle/working/image_mask_pairs.csv", index=False)

Dataset Path Verification:
Image folder : /kaggle/input/datasets/iamtapendu/chest-x-ray-lungs-segmentation/Chest-X-Ray/Chest-X-Ray/image
Mask folder  : /kaggle/input/datasets/iamtapendu/chest-x-ray-lungs-segmentation/Chest-X-Ray/Chest-X-Ray/mask
Image folder exists: True
Mask folder exists : True
Found 704 image files

✅ Total matched image-mask pairs: 704

First 5 pairs:
                                               image  \
0  /kaggle/input/datasets/iamtapendu/chest-x-ray-...   
1  /kaggle/input/datasets/iamtapendu/chest-x-ray-...   
2  /kaggle/input/datasets/iamtapendu/chest-x-ray-...   
3  /kaggle/input/datasets/iamtapendu/chest-x-ray-...   
4  /kaggle/input/datasets/iamtapendu/chest-x-ray-...   

                                                mask  
0  /kaggle/input/datasets/iamtapendu/chest-x-ray-...  
1  /kaggle/input/datasets/iamtapendu/chest-x-ray-...  
2  /kaggle/input/datasets/iamtapendu/chest-x-ray-...  
3  /kaggle/input/datasets/iamtapendu/chest-x-ray-...  
4  /kaggle/in

In [22]:
def get_transforms(mode="train", img_size=128):
    if mode == "train":
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.5),
            A.RandomBrightnessContrast(p=0.4),
            A.Normalize(mean=(0.5,), std=(0.5,)),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=(0.5,), std=(0.5,)),
            ToTensorV2(),
        ])

class ChestDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img = aug['image'].float()
            mask = aug['mask'].unsqueeze(0).float()
        return img, mask

In [23]:
# =============================================
# CELL 5: Corrected Dynamic UNet++ Model (Fixed Channel Mismatch)
# =============================================

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class UNetPlusPlus(nn.Module):
    def __init__(self, in_channels=1, num_classes=1, base_filters=32, depth=4, deep_supervision=True):
        super().__init__()
        self.depth = depth
        self.deep_supervision = deep_supervision
        
        # Encoder filters: 32, 64, 128, 256, ...
        self.filters = [base_filters * (2 ** i) for i in range(depth)]
        
        self.pool = nn.MaxPool2d(2, 2)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        
        # Encoder
        self.enc = nn.ModuleList()
        for i in range(depth):
            in_ch = in_channels if i == 0 else self.filters[i-1]
            self.enc.append(ConvBlock(in_ch, self.filters[i]))
        
        # Decoder
        self.dec = nn.ModuleList()
        for i in range(depth - 1):
            in_ch = self.filters[i+1] + self.filters[i]   # Skip connection concat
            self.dec.append(ConvBlock(in_ch, self.filters[i]))
        
        self.final = nn.Conv2d(self.filters[0], num_classes, kernel_size=1)

    def forward(self, x):
        skips = []
        
        # === Encoder ===
        for enc_block in self.enc:
            x = enc_block(x)
            skips.append(x)
            x = self.pool(x)
        
        # === Decoder ===
        skips = skips[::-1]  # Reverse skips
        
        for i, dec_block in enumerate(self.dec):
            x = self.up(x)
            # Resize skip feature map to match current x size
            skip = skips[i]
            if x.shape[-2:] != skip.shape[-2:]:
                skip = F.interpolate(skip, size=x.shape[-2:], mode='bilinear', align_corners=True)
            
            x = torch.cat([x, skip], dim=1)
            x = dec_block(x)
        
        output = self.final(x)
        return output

# ==================== TEST MODEL ====================
model_test = UNetPlusPlus(depth=4, base_filters=32).to(DEVICE)
x_test = torch.randn(2, 1, 128, 128).to(DEVICE)
out = model_test(x_test)

print(f"✅ Model Test Passed!")
print(f"Input Shape : {x_test.shape}")
print(f"Output Shape: {out.shape}")
print(f"Total Parameters: {sum(p.numel() for p in model_test.parameters()) / 1e6:.2f}M")

RuntimeError: Given groups=1, weight of size [32, 96, 3, 3], expected input[2, 512, 16, 16] to have 96 channels, but got 512 channels instead

In [ ]:
# =============================================
# CELL 6: Loss Functions, Metrics & Training Helpers
# =============================================

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        batch = probs.size(0)
        p = probs.view(batch, -1)
        t = targets.view(batch, -1)
        inter = (p * t).sum(1)
        dice = (2 * inter + self.smooth) / (p.sum(1) + t.sum(1) + self.smooth)
        return 1 - dice.mean()

class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce_weight = bce_weight
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
    
    def forward(self, logits, targets):
        return (self.bce_weight * self.bce(logits, targets) + 
                (1 - self.bce_weight) * self.dice(logits, targets))

# ==================== METRICS ====================
def dice_score(logits, targets, threshold=0.5, smooth=1.0):
    probs = (torch.sigmoid(logits) > threshold).float()
    batch = probs.size(0)
    p = probs.view(batch, -1)
    t = targets.view(batch, -1)
    inter = (p * t).sum(1)
    dice = (2 * inter + smooth) / (p.sum(1) + t.sum(1) + smooth)
    return dice.mean().item()

def iou_score(logits, targets, threshold=0.5, smooth=1.0):
    probs = (torch.sigmoid(logits) > threshold).float()
    batch = probs.size(0)
    p = probs.view(batch, -1)
    t = targets.view(batch, -1)
    inter = (p * t).sum(1)
    union = (p + t - p * t).sum(1)
    iou = (inter + smooth) / (union + smooth)
    return iou.mean().item()

def pixel_accuracy(logits, targets, threshold=0.5):
    probs = (torch.sigmoid(logits) > threshold).float()
    return (probs == targets).float().mean().item()

# ==================== TRAINING FUNCTIONS ====================
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = total_dice = total_iou = 0.0
    for imgs, masks in tqdm(loader, desc="Training", leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        
        optimizer.zero_grad()
        preds = model(imgs)
        
        if isinstance(preds, list):  # Deep Supervision
            loss = sum(criterion(p, masks) for p in preds) / len(preds)
            pred_final = preds[-1]
        else:
            loss = criterion(preds, masks)
            pred_final = preds
        
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        total_dice += dice_score(pred_final.detach(), masks)
        total_iou += iou_score(pred_final.detach(), masks)
    
    n = len(loader)
    return total_loss / n, total_dice / n, total_iou / n


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss = total_dice = total_iou = total_acc = 0.0
    
    for imgs, masks in tqdm(loader, desc="Validating", leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        preds = model(imgs)
        
        if isinstance(preds, list):
            loss = sum(criterion(p, masks) for p in preds) / len(preds)
            pred_final = preds[-1]
        else:
            loss = criterion(preds, masks)
            pred_final = preds
        
        total_loss += loss.item()
        total_dice += dice_score(pred_final, masks)
        total_iou += iou_score(pred_final, masks)
        total_acc += pixel_accuracy(pred_final, masks)
    
    n = len(loader)
    return total_loss / n, total_dice / n, total_iou / n, total_acc / n

print("✅ Loss, Metrics & Training Functions Loaded Successfully!")

In [ ]:
ablation_results = []

for exp in CFG.experiments:
    print(f"\n{'='*20} Experiment {exp['exp']}: {exp['name']} {'='*20}")
    
    CFG.img_size = exp['img_size']
    depth = exp['depth']
    
    # Recreate transforms and datasets with new size
    train_ds = ChestDataset(pairs[:int(0.8*len(pairs))], get_transforms("train", CFG.img_size))
    val_ds = ChestDataset(pairs[int(0.8*len(pairs)):], get_transforms("val", CFG.img_size))
    
    train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    model = UNetPlusPlus(in_channels=1, num_classes=1, base_filters=32, depth=depth).to(DEVICE)
    
    optimizer = AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
    criterion = BCEDiceLoss()
    
    best_dice = 0
    for epoch in range(CFG.num_epochs):
        train_one_epoch(model, train_loader, optimizer, criterion)
        _, val_dice, val_iou, _ = validate(model, val_loader, criterion)
        
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), f"{CFG.save_dir}/best_model_exp{exp['exp']}.pth")
    
    ablation_results.append({
        "Exp": exp['exp'],
        "PatchSize": f"{exp['img_size']}x{exp['img_size']}",
        "Depth": f"{depth}-{depth}",
        "MeanIoU": round(best_dice*100, 2),   # placeholder
        "F1Score": round(best_dice*100, 2),
        # Add other metrics as needed
    })

# Save Ablation Table
df_ablation = pd.DataFrame(ablation_results)
df_ablation.to_csv("/kaggle/working/PatchSize_Depth_Ablation.csv", index=False)
display(df_ablation)